In [1]:
from transformers import AutoImageProcessor
import torch.optim as optim
import webdataset as wds
from pathlib import Path
from modules import *
import csv
import argparse
import os

In [2]:
DATASET_PATH = "/home/austen/GeoDataset/dataset_sharded"
S2_LABELS_DIR = Path(DATASET_PATH) / "s2_labels"
BATCH_SIZE = 2
WORKERS = 2
S2_LEVELS = list(range(3, 7))
PRETRAINED_MODEL_ID = "facebook/convnext-base-384"
CHECKPOINT_PATH = "checkpoints/checkpoint_1235.pt"
DEVICE = "cuda"
RESIZE = 384
IMAGES = 10000

In [3]:
processor = AutoImageProcessor.from_pretrained(PRETRAINED_MODEL_ID, use_fast=True)
processor.do_resize = True
processor.size = {"shortest_edge": RESIZE}

s2_labels_dir = Path(DATASET_PATH) / "s2_labels"
idx2id, id2idx, _ = build_s2_index_maps(s2_labels_dir, S2_LEVELS)
parent_tables = build_parent_tables_from_maps(idx2id, id2idx, S2_LEVELS)

dataset = GeoWebDataset(
    DATASET_PATH,
    processor,
    levels=S2_LEVELS,
    shuffle=True,
    num_shards_limit=None,
    id2idx=id2idx,
)

model = HierarchicalConvNeXt(
    pretrained_name=PRETRAINED_MODEL_ID,
    num_classes=dataset.num_classes_list[-1],
    freeze=False,
)

loader = wds.WebLoader(
    dataset.dataset,
    num_workers=WORKERS,
    batch_size=BATCH_SIZE,
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True,
)

In [4]:
evaluator = Evaluator(
    model=model, 
    loader=loader, 
    s2_levels=S2_LEVELS, 
    parent_maps=parent_tables,
    num_classes_per_level=dataset.num_classes_list, 
    idx2id=idx2id
).to(DEVICE)

evaluator.load_checkpoint(CHECKPOINT_PATH)

out = evaluator(max_batches = IMAGES // BATCH_SIZE)
out

Epoch: 63


Evaluating: 100%|██████████| 5000/5000 [02:35<00:00, 32.08it/s]


,Top-k,S2 level,Accuracy,Random baseline
0,1,3,0.4398,0.0058
1,1,4,0.3208,0.0022
2,1,5,0.2217,0.0008
3,1,6,0.1477,0.0003
4,5,3,0.8209,0.0058
5,5,4,0.6719,0.0022
6,5,5,0.4987,0.0008
7,5,6,0.3479,0.0003
8,10,3,0.9171,0.0058
9,10,4,0.8082,0.0022
